# 🧠 Brain Tumor Classification using EfficientNetB2

This notebook classifies brain MRI images into **4 categories**:
- Glioma
- Meningioma
- No Tumor
- Pituitary

Using **EfficientNetB2** (transfer learning from ImageNet) with a two-phase training strategy.

## 1. Important Libraries

In [ ]:
# Check GPU first
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

if tf.config.list_physical_devices('GPU'):
    print("GPU Name:", tf.config.list_physical_devices('GPU')[0].name)
else:
    print("⚠  No GPU detected — training will be very slow!")

In [ ]:
# Importing necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import shutil
import random
import math

# sklearn packages
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# keras / tensorflow packages
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.applications import EfficientNetB2
from keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from keras import layers
from keras.models import Model, Sequential
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization

## 2. Download & Setup Dataset

In [ ]:
# Upload your kaggle.json to Colab first, then run this cell

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!unzip -q brain-tumor-mri-dataset.zip -d brain_tumor_data

print("\n✅ Dataset downloaded and extracted!")

In [ ]:
# Set dataset paths
TRAIN_DIR = 'brain_tumor_data/Training'
TEST_DIR  = 'brain_tumor_data/Testing'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50

# Verify the dataset structure
print("Training classes:", os.listdir(TRAIN_DIR))
print("Testing classes:", os.listdir(TEST_DIR))

# Count images per class
print("\n--- Training Set ---")
total_train = 0
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        total_train += count
        print(f"  {cls}: {count} images")
print(f"  Total: {total_train} images")

print("\n--- Testing Set ---")
total_test = 0
for cls in sorted(os.listdir(TEST_DIR)):
    cls_path = os.path.join(TEST_DIR, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        total_test += count
        print(f"  {cls}: {count} images")
print(f"  Total: {total_test} images")

## 3. Preprocess the Data

### 3.1 Check Image Size Distribution

Before resizing all images, let's check the original image dimensions across the dataset.

In [ ]:
# Check the distribution of image sizes in the training and test sets
train_heights = []
train_widths = []
test_heights = []
test_widths = []

# Scan training images
for root, _, files in os.walk(TRAIN_DIR):
    for file in files:
        if file.startswith('.'):
            continue
        img_path = os.path.join(root, file)
        img = cv2.imread(img_path)
        if img is not None:
            h, w = img.shape[:2]
            train_heights.append(h)
            train_widths.append(w)

# Scan testing images
for root, _, files in os.walk(TEST_DIR):
    for file in files:
        if file.startswith('.'):
            continue
        img_path = os.path.join(root, file)
        img = cv2.imread(img_path)
        if img is not None:
            h, w = img.shape[:2]
            test_heights.append(h)
            test_widths.append(w)

print(f"Training images scanned: {len(train_heights)}")
print(f"Testing images scanned:  {len(test_heights)}")

In [ ]:
# Plot image size distributions as scatter plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
axes[0].scatter(train_widths, train_heights, alpha=0.3, color='#4ECDC4', edgecolors='none', s=20)
axes[0].set_title('Training Set — Image Size Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Height (pixels)')
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(test_widths, test_heights, alpha=0.3, color='#FF6B6B', edgecolors='none', s=20)
axes[1].set_title('Testing Set — Image Size Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Width (pixels)')
axes[1].set_ylabel('Height (pixels)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print stats
print(f"\nTraining — Width:  min={min(train_widths)}, max={max(train_widths)}, mean={np.mean(train_widths):.0f}")
print(f"Training — Height: min={min(train_heights)}, max={max(train_heights)}, mean={np.mean(train_heights):.0f}")
print(f"Testing  — Width:  min={min(test_widths)}, max={max(test_widths)}, mean={np.mean(test_widths):.0f}")
print(f"Testing  — Height: min={min(test_heights)}, max={max(test_heights)}, mean={np.mean(test_heights):.0f}")
print(f"\n→ All images will be resized to {IMG_SIZE} for the model.")

### 3.2 Load Raw Images & Inspect Shapes

Load a sample of images to verify pixel values, channels, and shapes before passing them to the model.

In [ ]:
# Load sample images from each class to inspect
sample_images = []
sample_labels = []

for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    files = os.listdir(cls_path)
    # Pick 3 random samples from each class
    for fname in random.sample(files, min(3, len(files))):
        img_path = os.path.join(cls_path, fname)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img, IMG_SIZE)
            sample_images.append(img_resized)
            sample_labels.append(cls)

sample_images = np.array(sample_images)

print(f"Loaded {len(sample_images)} sample images")
print(f"Shape of sample array: {sample_images.shape}")
print(f"Pixel value range: [{sample_images.min()}, {sample_images.max()}]")
print(f"Data type: {sample_images.dtype}")
print(f"Single image shape: {sample_images[0].shape}")

### 3.3 Check for Corrupted / Invalid Images

In [ ]:
# Scan for corrupted images that OpenCV cannot read
corrupted_files = []
grayscale_files = []
total_checked = 0

for split_name, split_dir in [('Training', TRAIN_DIR), ('Testing', TEST_DIR)]:
    for root, _, files in os.walk(split_dir):
        for fname in files:
            if fname.startswith('.'):
                continue
            total_checked += 1
            fpath = os.path.join(root, fname)
            img = cv2.imread(fpath)
            if img is None:
                corrupted_files.append(fpath)
            elif len(img.shape) == 2 or img.shape[2] == 1:
                grayscale_files.append(fpath)

print(f"Total images checked: {total_checked}")
print(f"Corrupted images:     {len(corrupted_files)}")
print(f"Grayscale images:     {len(grayscale_files)}")

if corrupted_files:
    print("\n⚠ Corrupted files:")
    for f in corrupted_files[:10]:
        print(f"  {f}")
else:
    print("\n✅ No corrupted images found!")

if grayscale_files:
    print(f"\nℹ  {len(grayscale_files)} grayscale images found (will be converted to RGB by the generator)")

## 4. Distribution of Classes & Sample Images

In [ ]:
# Count samples per class in Training set
train_class_counts = {}
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if os.path.isdir(cls_path):
        train_class_counts[cls] = len(os.listdir(cls_path))

# Count samples per class in Testing set
test_class_counts = {}
for cls in sorted(os.listdir(TEST_DIR)):
    cls_path = os.path.join(TEST_DIR, cls)
    if os.path.isdir(cls_path):
        test_class_counts[cls] = len(os.listdir(cls_path))

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

# Training distribution
bars1 = axes[0].bar(train_class_counts.keys(), train_class_counts.values(), color=colors)
axes[0].set_title('Training Set — Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Images')
for bar, count in zip(bars1, train_class_counts.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                 str(count), ha='center', va='bottom', fontsize=11)

# Testing distribution
bars2 = axes[1].bar(test_class_counts.keys(), test_class_counts.values(), color=colors)
axes[1].set_title('Testing Set — Class Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Number of Images')
for bar, count in zip(bars2, test_class_counts.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                 str(count), ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Plot sample images from each class
classes = sorted(os.listdir(TRAIN_DIR))
num_classes = len(classes)
samples_per_class = 3

fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(12, 3 * num_classes))

for row, cls in enumerate(classes):
    cls_path = os.path.join(TRAIN_DIR, cls)
    all_files = os.listdir(cls_path)
    chosen = random.sample(all_files, min(samples_per_class, len(all_files)))

    for col, fname in enumerate(chosen):
        img_path = os.path.join(cls_path, fname)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        axes[row][col].imshow(img)
        axes[row][col].set_title(f"{cls} ({img.shape[1]}x{img.shape[0]})", fontsize=10)
        axes[row][col].axis('off')

plt.suptitle('Sample Images from Each Class (Original Size)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Data Augmentation

Data augmentation is a technique in machine learning used to artificially increase the size and diversity of a training dataset by creating modified versions of existing data.

Instead of collecting new data (which is expensive and time-consuming), we apply **transformations** to the current data to teach the model that the core features remain the same even if the appearance changes.

**Why It Matters:**
- **Prevents Overfitting:** It stops the model from "memorizing" specific training examples.
- **Improves Generalization:** It helps the model perform better on new, unseen data by exposing it to more variations.
- **Cost-Effective:** We get a larger dataset for "free" without manual labeling.

**Important Note about EfficientNet:**
> EfficientNet has its own built-in preprocessing/normalization. We must use `efficientnet_preprocess` instead of `rescale=1./255`. Using `rescale=1./255` would **double-scale** the data (scaling to [0,1] and then EfficientNet scales again internally), which destroys model accuracy.

In [ ]:
# Visualize what data augmentation looks like on a sample image

# Pick a random training image
sample_cls = random.choice(sorted(os.listdir(TRAIN_DIR)))
sample_cls_path = os.path.join(TRAIN_DIR, sample_cls)
sample_fname = random.choice(os.listdir(sample_cls_path))
sample_img = cv2.imread(os.path.join(sample_cls_path, sample_fname))
sample_img = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
sample_img = cv2.resize(sample_img, IMG_SIZE)

# Create a temporary augmentation generator (without preprocessing for visualization)
aug_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    brightness_range=(0.8, 1.2),
    fill_mode='nearest'
)

# Generate augmented versions
sample_batch = np.expand_dims(sample_img, axis=0)
aug_iter = aug_datagen.flow(sample_batch, batch_size=1)

# Plot original + 7 augmented versions
plt.figure(figsize=(16, 4))

plt.subplot(1, 8, 1)
plt.imshow(sample_img)
plt.title(f'Original\n({sample_cls})', fontsize=9)
plt.axis('off')

for i in range(7):
    aug_img = next(aug_iter)[0].astype(np.uint8)
    plt.subplot(1, 8, i + 2)
    plt.imshow(aug_img)
    plt.title(f'Augmented {i+1}', fontsize=9)
    plt.axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Create the actual data generators for training

# Training data generator with augmentation + EfficientNet preprocessing
train_datagen = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.2
)

# Validation/Test generator — NO augmentation, only EfficientNet preprocessing
val_test_datagen = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess
)

# Training split (80% of Training folder)
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

# Validation split (20% of Training folder)
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

# Test set (separate Testing folder)
test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Get class names from generator (correct order guaranteed)
class_indices = train_generator.class_indices
class_names = {v: k for k, v in class_indices.items()}
NUM_CLASSES = len(class_names)

print(f"\nClass mapping    : {class_indices}")
print(f"Training samples : {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Test samples     : {test_generator.samples}")
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
# Visualize a batch from the training generator (after preprocessing)
images, labels = next(train_generator)
n_samples = min(8, len(images))

plt.figure(figsize=(12, 6))
for i in range(n_samples):
    plt.subplot(2, 4, i + 1)
    # Undo EfficientNet preprocessing for display
    img = images[i].copy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    plt.imshow(img)
    label_idx = np.argmax(labels[i])
    plt.title(class_names[label_idx], fontsize=10)
    plt.axis('off')

plt.suptitle('Training Batch — After Augmentation & Preprocessing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print the preprocessed pixel value range
print(f"Preprocessed pixel range: [{images.min():.2f}, {images.max():.2f}]")
print(f"Batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

## 6. Model Building — EfficientNetB2

In [ ]:
# Load the pre-trained EfficientNetB2 base model
eff_base_model = EfficientNetB2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze ALL base model layers initially
# We don't want to ruin the pre-trained weights during initial training
eff_base_model.trainable = False

# Build custom classifier head on top
eff_model = Sequential([
    keras.Input(shape=(224, 224, 3)),
    eff_base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dropout(0.3),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

eff_model.summary()

In [ ]:
# Print all layers in the EfficientNetB2 base model
# (useful for finding the last conv layer name for Grad-CAM later)
print(f"Total layers in EfficientNetB2: {len(eff_base_model.layers)}")
print("\nLast 20 layers:")
for layer in eff_base_model.layers[-20:]:
    print(f"  {layer.name:40s} | {layer.__class__.__name__:25s} | trainable={layer.trainable}")

## 7. Model Training

### 7.1 Compile the Model

In [ ]:
eff_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model compiled!")
print(f"Optimizer: Adam (lr=1e-4)")
print(f"Loss: categorical_crossentropy")
print(f"Metrics: accuracy")

In [ ]:
# Setup callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_brain_tumor_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("✅ Callbacks ready!")

### 7.2 Phase 1 — Train Classification Head (Base Frozen)

In this phase, all EfficientNetB2 layers are frozen. Only the custom classification head trains on the brain tumor data.

In [ ]:
print("="*60)
print("  Phase 1: Training classification head (base frozen)")
print("="*60)

history = eff_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    verbose=1
)

### 7.3 Phase 1 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Phase 1 — Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Phase 1 — Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print best values
best_epoch = np.argmax(history.history['val_accuracy'])
print(f"\nBest Epoch: {best_epoch + 1}")
print(f"Best Val Accuracy: {history.history['val_accuracy'][best_epoch]:.4f}")
print(f"Best Val Loss: {history.history['val_loss'][best_epoch]:.4f}")

### 7.4 Phase 2 — Fine-Tune Top Layers

Now we unfreeze the last 40 layers of EfficientNetB2 and train with a much smaller learning rate to avoid catastrophic forgetting.

In [ ]:
print("="*60)
print("  Phase 2: Fine-tuning top base-model layers")
print("="*60)

# Unfreeze the last 40 layers
eff_base_model.trainable = True
for layer in eff_base_model.layers[:-40]:
    layer.trainable = False

# Count trainable vs frozen layers
trainable_count = sum(1 for layer in eff_base_model.layers if layer.trainable)
frozen_count = sum(1 for layer in eff_base_model.layers if not layer.trainable)
print(f"Trainable layers: {trainable_count}")
print(f"Frozen layers:    {frozen_count}")

# Re-compile with smaller learning rate
eff_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# New callbacks for fine-tuning
fine_tune_early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

fine_tune_reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-8,
    verbose=1
)

FINE_TUNE_EPOCHS = 20

history_fine = eff_model.fit(
    train_generator,
    epochs=FINE_TUNE_EPOCHS,
    validation_data=val_generator,
    callbacks=[fine_tune_early_stopping, fine_tune_reduce_lr, checkpoint],
    verbose=1
)

### 7.5 Phase 2 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history_fine.history['accuracy'], label='Train Accuracy')
axes[0].plot(history_fine.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Phase 2 (Fine-tune) — Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history_fine.history['loss'], label='Train Loss')
axes[1].plot(history_fine.history['val_loss'], label='Val Loss')
axes[1].set_title('Phase 2 (Fine-tune) — Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print best values
best_epoch_ft = np.argmax(history_fine.history['val_accuracy'])
print(f"\nBest Epoch: {best_epoch_ft + 1}")
print(f"Best Val Accuracy: {history_fine.history['val_accuracy'][best_epoch_ft]:.4f}")
print(f"Best Val Loss: {history_fine.history['val_loss'][best_epoch_ft]:.4f}")

## 8. Evaluation

In [ ]:
# Evaluate on Test Set
print("="*60)
print("  Test Set Evaluation")
print("="*60)

test_loss, test_accuracy = eff_model.evaluate(test_generator, verbose=1)
print(f"\nTest Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## 9. Prediction & Confusion Matrix

In [ ]:
# Get predictions on test set
predictions = eff_model.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes

# Class names in correct order
target_names = [class_names[i] for i in range(NUM_CLASSES)]

# Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names,
            yticklabels=target_names)
plt.title('Confusion Matrix — EfficientNetB2', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
report = classification_report(true_classes, predicted_classes, target_names=target_names)
print("\nClassification Report:")
print("=" * 60)
print(report)

## 10. Grad-CAM Visualization

Grad-CAM (Gradient-weighted Class Activation Mapping) is the "X-ray vision" of Deep Learning models. It highlights which regions of the MRI the model focuses on to make its prediction.

> The **red/yellow** areas in the heatmap show the regions the model considers most important for classification. **Blue** areas have less influence on the prediction.

In [ ]:
# Find the last Conv2D layer name in EfficientNetB2 for Grad-CAM
last_conv_layer = None
for layer in reversed(eff_base_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer = layer.name
        break

print(f"Last conv layer for Grad-CAM: {last_conv_layer}")
print(f"Total layers in base model: {len(eff_base_model.layers)}")

In [ ]:
# Get a batch of test images for Grad-CAM visualization
test_images, test_labels = next(test_generator)
num_display = min(8, len(test_images))

# Get the base model from the Sequential model
base = eff_model.layers[1]

plt.figure(figsize=(16, 4 * math.ceil(num_display / 4)))

for i in range(num_display):
    img = test_images[i]
    img_batch = np.expand_dims(img, axis=0)

    # --- Generate Grad-CAM heatmap ---
    grad_model = tf.keras.models.Model(
        inputs=base.input,
        outputs=[base.get_layer(last_conv_layer).output, base.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_batch)
        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs_val = conv_outputs[0]
    heatmap = conv_outputs_val @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    heatmap = heatmap.numpy()

    # --- Overlay heatmap on image ---
    display_img = img.copy()
    display_img = ((display_img - display_img.min()) / (display_img.max() - display_img.min() + 1e-8) * 255).astype(np.uint8)

    heatmap_resized = cv2.resize(heatmap, (display_img.shape[1], display_img.shape[0]))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

    superimposed = cv2.addWeighted(display_img, 0.6, heatmap_colored, 0.4, 0)

    # --- Plot ---
    pred_class = np.argmax(eff_model.predict(img_batch, verbose=0)[0])
    true_class = np.argmax(test_labels[i])

    plt.subplot(math.ceil(num_display / 4), 4, i + 1)
    plt.imshow(superimposed)
    color = 'green' if pred_class == true_class else 'red'
    plt.title(f"True: {class_names[true_class]}\nPred: {class_names[pred_class]}",
              fontsize=9, color=color)
    plt.axis('off')

plt.suptitle('Grad-CAM Visualization — EfficientNetB2', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Save the Trained Model

In [ ]:
eff_model.save('EfficientNetB2_brain_tumor_model.keras')
print("✅ Model saved to: EfficientNetB2_brain_tumor_model.keras")

## 12. Download Model (Google Colab)

In [ ]:
from google.colab import files
files.download('EfficientNetB2_brain_tumor_model.keras')
files.download('best_brain_tumor_model.keras')

print("\n✅ All done! Training complete.")